In [1]:
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma

In [2]:
from langchain_core.documents import Document

# Create LangChain documents for IPL players

doc1 = Document(
        page_content="Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.",
        metadata={"team": "Royal Challengers Bangalore"}
    )
doc2 = Document(
        page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
        metadata={"team": "Mumbai Indians"}
    )
doc3 = Document(
        page_content="MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.",
        metadata={"team": "Chennai Super Kings"}
    )
doc4 = Document(
        page_content="Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.",
        metadata={"team": "Mumbai Indians"}
    )
doc5 = Document(
        page_content="Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.",
        metadata={"team": "Chennai Super Kings"}
    )

In [3]:
# Combine all the documents
docs = [doc1, doc2, doc3, doc4, doc5]

In [7]:
vector_store = Chroma(
    embedding_function=OllamaEmbeddings(model="embeddinggemma:latest"),
    persist_directory="./chroma_db",
    collection_name="ipl_players"
)

In [8]:
# Add the documents to the vector store each doc will have a unique id
vector_store.add_documents(docs)

['59a1cd9c-3c9f-4939-af61-390f8bf1e421',
 'eef5e8c7-0354-4281-9ded-2a107e0468ab',
 'fcc39e07-7615-4a40-a914-8e827c9198cd',
 '9f370fe8-c037-496f-b5ef-d818044bd7cb',
 '9d3df7c0-437f-448f-93e7-966a3029bd66']

In [9]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['59a1cd9c-3c9f-4939-af61-390f8bf1e421',
  'eef5e8c7-0354-4281-9ded-2a107e0468ab',
  'fcc39e07-7615-4a40-a914-8e827c9198cd',
  '9f370fe8-c037-496f-b5ef-d818044bd7cb',
  '9d3df7c0-437f-448f-93e7-966a3029bd66'],
 'embeddings': array([[-0.04829332, -0.02939798, -0.0564089 , ...,  0.05658666,
          0.01583169, -0.00717874],
        [-0.07502022, -0.01692122, -0.0213124 , ...,  0.06940156,
         -0.02617546, -0.0384788 ],
        [-0.08895794, -0.01867576, -0.0274267 , ...,  0.03055765,
         -0.03483181,  0.01279135],
        [-0.06483486, -0.05916816, -0.03760871, ...,  0.04949442,
         -0.02733131, -0.0400927 ],
        [-0.049484  , -0.02581017, -0.02742201, ...,  0.05557529,
          0.00540692,  0.00134522]], shape=(5, 768)),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the mo

In [11]:
# search documents
vector_store.similarity_search(
    query='Who among these are a bowler?',
    k=2
)

[Document(id='9f370fe8-c037-496f-b5ef-d818044bd7cb', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
 Document(id='9d3df7c0-437f-448f-93e7-966a3029bd66', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.')]

In [12]:
# search with similarity score
# Smaller the score the shorter the distance from the query embedding to the document embedding and hence more relevant the document is to the query
vector_store.similarity_search_with_score(
    query='Who among these are a bowler?',
    k=2
)

[(Document(id='9f370fe8-c037-496f-b5ef-d818044bd7cb', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
  1.215400218963623),
 (Document(id='9d3df7c0-437f-448f-93e7-966a3029bd66', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  1.268464207649231)]

In [13]:
# meta-data filtering
vector_store.similarity_search_with_score(
    query="",
    filter={"team": "Chennai Super Kings"}
)

[(Document(id='fcc39e07-7615-4a40-a914-8e827c9198cd', metadata={'team': 'Chennai Super Kings'}, page_content='MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.'),
  1.5375335216522217),
 (Document(id='9d3df7c0-437f-448f-93e7-966a3029bd66', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  1.5409282445907593)]

In [14]:
# update documents
updated_doc1 = Document(
    page_content="Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple centuries in a single season. Despite RCB not winning an IPL title under his captaincy, Kohli's passion and fitness set a benchmark for the league. His ability to chase targets and anchor innings has made him one of the most dependable players in T20 cricket.",
    metadata={"team": "Royal Challengers Bangalore"}
)

vector_store.update_document(document_id='09a39dc6-3ba6-4ea7-927e-fdda591da5e4', document=updated_doc1)

In [15]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['59a1cd9c-3c9f-4939-af61-390f8bf1e421',
  'eef5e8c7-0354-4281-9ded-2a107e0468ab',
  'fcc39e07-7615-4a40-a914-8e827c9198cd',
  '9f370fe8-c037-496f-b5ef-d818044bd7cb',
  '9d3df7c0-437f-448f-93e7-966a3029bd66'],
 'embeddings': array([[-0.04829332, -0.02939798, -0.0564089 , ...,  0.05658666,
          0.01583169, -0.00717874],
        [-0.07502022, -0.01692122, -0.0213124 , ...,  0.06940156,
         -0.02617546, -0.0384788 ],
        [-0.08895794, -0.01867576, -0.0274267 , ...,  0.03055765,
         -0.03483181,  0.01279135],
        [-0.06483486, -0.05916816, -0.03760871, ...,  0.04949442,
         -0.02733131, -0.0400927 ],
        [-0.049484  , -0.02581017, -0.02742201, ...,  0.05557529,
          0.00540692,  0.00134522]], shape=(5, 768)),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the mo

In [16]:
# delete document
vector_store.delete(ids=['09a39dc6-3ba6-4ea7-927e-fdda591da5e4'])

In [17]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['59a1cd9c-3c9f-4939-af61-390f8bf1e421',
  'eef5e8c7-0354-4281-9ded-2a107e0468ab',
  'fcc39e07-7615-4a40-a914-8e827c9198cd',
  '9f370fe8-c037-496f-b5ef-d818044bd7cb',
  '9d3df7c0-437f-448f-93e7-966a3029bd66'],
 'embeddings': array([[-0.04829332, -0.02939798, -0.0564089 , ...,  0.05658666,
          0.01583169, -0.00717874],
        [-0.07502022, -0.01692122, -0.0213124 , ...,  0.06940156,
         -0.02617546, -0.0384788 ],
        [-0.08895794, -0.01867576, -0.0274267 , ...,  0.03055765,
         -0.03483181,  0.01279135],
        [-0.06483486, -0.05916816, -0.03760871, ...,  0.04949442,
         -0.02733131, -0.0400927 ],
        [-0.049484  , -0.02581017, -0.02742201, ...,  0.05557529,
          0.00540692,  0.00134522]], shape=(5, 768)),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the mo